# Chapter 21 — Volatility Modeling: Local, Stochastic, Jump, and Rough

Every model in the chapter, built from its own definition and then made to answer
the chapter's questions. **NumPy and SciPy only.** The Heston characteristic
function, the Dupire formula, Hagan's expansion, the Merton series and the rough
Bergomi hybrid scheme are written out rather than called from a library, so the
equation in the text and the code are the same object.

| Section | What is built | Method | Text |
|---|---|---|---|
| 1 | Black–Scholes and implied volatility | closed form, Brent | — |
| 2 | Heston | characteristic function, Gauss–Legendre | (21.4) |
| 3 | What each parameter does | smiles and their fitted statistics | Fig. 21.2, Table 21.2 |
| 4 | Local volatility | Dupire, by finite difference | (21.2), Ex. 21.1 |
| 5 | The forward smile | skew of the local-volatility function | Ex. 21.2 |
| 6 | The skew term structure | Heston against the power law | Fig. 21.4 |
| 7 | SABR | Hagan's expansion, and where its density goes negative | Section 21.4, (29.21)–(29.22) |
| 8 | Jumps | Merton's series | Ex. 21.3 |
| 9 | Rough volatility | rough Bergomi, hybrid scheme | (21.8), (21.9) |
| 10 | Local-stochastic volatility | leverage function by the particle method | (21.10), Ex. 21.5, Fig. 21.6 |
| 11 | Model risk | barriers under common random numbers | Ex. 21.7 |
| 12 | The calibration objective | two fits of one surface | Ex. 21.6 |

Throughout, `S0 = 100`, `r = q = 0`, and the base case is the chapter's:
`v0 = theta = 0.04`, `kappa = 1.5`, `sigma_v = 0.6`, `rho = -0.7`.

Monte Carlo results are compared **in standard errors**, never digit by digit.

## 1. Black–Scholes–Merton and implied volatility

Everything below is quoted as an implied volatility, so the inversion has to come
first. Brent on the price is slower than a Newton iteration and completely
reliable, which is the right trade for a notebook.

In [1]:
import numpy as np
from math import log, exp, sqrt, pi, factorial
from scipy.stats import norm
from scipy.optimize import brentq, least_squares

S0, R, Q = 100.0, 0.0, 0.0
BASE = dict(v0=0.04, kappa=1.5, theta=0.04, sigv=0.6, rho=-0.7)


def bs_call(S, K, r, q, sig, T):
    if sig <= 0 or T <= 0:
        return max(S * exp(-q * T) - K * exp(-r * T), 0.0)
    d1 = (log(S / K) + (r - q + 0.5 * sig * sig) * T) / (sig * sqrt(T))
    d2 = d1 - sig * sqrt(T)
    return S * exp(-q * T) * norm.cdf(d1) - K * exp(-r * T) * norm.cdf(d2)


def bs_vega(S, K, r, q, sig, T):
    d1 = (log(S / K) + (r - q + 0.5 * sig * sig) * T) / (sig * sqrt(T))
    return S * exp(-q * T) * norm.pdf(d1) * sqrt(T)


def implied_vol(price, S, K, r, q, T):
    """Brent between 0.01% and 500% volatility."""
    lo, hi = 1e-6, 5.0
    if price <= bs_call(S, K, r, q, lo, T) + 1e-14:
        return float("nan")
    return brentq(lambda s: bs_call(S, K, r, q, s, T) - price, lo, hi, xtol=1e-12)


print("round trip: price a 20% one-year call and invert it")
p = bs_call(S0, 100.0, R, Q, 0.20, 1.0)
print("  price %.6f   implied back %.8f" % (p, implied_vol(p, S0, 100.0, R, Q, 1.0)))

round trip: price a 20% one-year call and invert it
  price 7.965567   implied back 0.20000000


## 2. Heston

Equation (21.4) is a pair of correlated diffusions,

$$dS = (r-q)S\,dt + \sqrt{v}\,S\,dW_1, \qquad
  dv = \kappa(\theta-v)\,dt + \sigma_v\sqrt{v}\,dW_2, \qquad dW_1dW_2 = \rho\,dt$$

and its whole practical appeal is that the characteristic function of $\log S_T$
is available, so a price is one numerical integral rather than a simulation:

$$C = S_0e^{-qT}P_1 - Ke^{-rT}P_2, \qquad
  P_j = \tfrac12 + \frac{1}{\pi}\int_0^\infty
  \mathrm{Re}\!\left[\frac{e^{-iu\ln K}\varphi_j(u)}{iu}\right]du$$

Two implementation points that are not optional. The square root in the
characteristic function must be taken on the branch that keeps
$\log((1-ge^{-dT})/(1-g))$ continuous, which avoids the *little Heston trap*; the form below
is the stable one. And the integrand is smooth and decays fast, so a fixed
**Gauss–Legendre** rule on $[0, 150]$ prices a whole row of strikes at once and to
machine precision, which is what makes the Dupire differencing in §4 possible.

The sanity check is the one that matters: as $\sigma_v \to 0$ the model must
collapse to Black–Scholes–Merton at $\sqrt{v_0}$.

In [2]:
GL_N, UMAX = 512, 150.0
_u, _w = np.polynomial.legendre.leggauss(GL_N)
U = 0.5 * (_u + 1.0) * UMAX
W = _w * 0.5 * UMAX


def heston_cf(u, T, p):
    """E[exp(i u log S_T)] on the continuous branch."""
    iu = 1j * u
    b = p["kappa"] - p["rho"] * p["sigv"] * iu
    d = np.sqrt(b * b + p["sigv"] ** 2 * (iu + u * u))
    g = (b - d) / (b + d)
    e = np.exp(-d * T)
    C = (p["kappa"] * p["theta"] / p["sigv"] ** 2) * (
        (b - d) * T - 2.0 * np.log((1 - g * e) / (1 - g)))
    D = (p["v0"] / p["sigv"] ** 2) * (b - d) * (1 - e) / (1 - g * e)
    return np.exp(iu * (log(S0) + (R - Q) * T) + C + D)


def heston_call(Ks, T, **kw):
    """Vectorized over strikes. Returns an array."""
    p = dict(BASE); p.update(kw)
    Ks = np.atleast_1d(np.asarray(Ks, float))
    lnK = np.log(Ks)[:, None]
    fwd = S0 * exp((R - Q) * T)
    e = np.exp(-1j * U * lnK)
    P1 = 0.5 + (W * np.real(e * heston_cf(U - 1j, T, p) / (1j * U * fwd))).sum(1) / pi
    P2 = 0.5 + (W * np.real(e * heston_cf(U, T, p) / (1j * U))).sum(1) / pi
    return S0 * exp(-Q * T) * P1 - Ks * exp(-R * T) * P2


def heston_iv(K, T, **kw):
    return implied_vol(float(heston_call(K, T, **kw)[0]), S0, float(K), R, Q, T)


print("limit check: sigma_v -> 0 must give Black-Scholes at sqrt(v0) = 20%")
print("  Heston   %.10f" % heston_call(100.0, 1.0, sigv=1e-6, rho=0.0)[0])
print("  BS(20%%)  %.10f" % bs_call(S0, 100.0, R, Q, 0.20, 1.0))
print()
print("base case, one year")
print("  ATM call %.4f    ATM implied volatility %.2f%%"
      % (heston_call(100.0, 1.0)[0], 100 * heston_iv(100.0, 1.0)))

limit check: sigma_v -> 0 must give Black-Scholes at sqrt(v0) = 20%
  Heston   7.9655612546
  BS(20%)  7.9655674554

base case, one year
  ATM call 6.7509    ATM implied volatility 16.94%


## 3. What each parameter does — Figure 21.2 and Table 21.2

Two statistics summarize a smile, both measured in log-moneyness
$k = \log(K/F)$ by central differences at $k = \pm 0.10$:

* **skew** $\;\partial\sigma/\partial k$ — the tilt, set mainly by $\rho$;
* **curvature** $\;\partial^2\sigma/\partial k^2$ — the smile's bend, set mainly by $\sigma_v$.

The point of the table is the interaction. Raising $\sigma_v$ does not raise
curvature alone: it steepens the skew and drops the at-the-money level too, which
is why calibration is a genuine optimization rather than three one-dimensional
fits.

In [3]:
KS_FIG = np.array([70., 80., 90., 100., 110., 120., 130.])

print("Figure 21.2, one-year implied volatility (%)")
print("  %-16s %s" % ("", "".join("%8.0f" % K for K in KS_FIG)))
for tag, kw in [("rho=-0.9", dict(rho=-0.9)), ("rho=-0.5", dict(rho=-0.5)),
                ("rho=0", dict(rho=0.0)), ("sigv=0.2", dict(sigv=0.2)),
                ("sigv=0.6 (base)", {}), ("sigv=1.0", dict(sigv=1.0))]:
    row = [100 * heston_iv(K, 1.0, **kw) for K in KS_FIG]
    print("  %-16s %s" % (tag, "".join("%8.2f" % x for x in row)))


def smile_stats(T=1.0, h=0.10, **kw):
    """ATM level, skew and curvature in log-moneyness, central differences at +/- h."""
    F = S0 * exp((R - Q) * T)
    a, b, c = (heston_iv(F * exp(-h), T, **kw), heston_iv(F, T, **kw),
               heston_iv(F * exp(h), T, **kw))
    return 100 * b, (c - a) / (2 * h), (c - 2 * b + a) / h ** 2


print()
print("Table 21.2, one year")
print("  %-28s %9s %9s %11s" % ("", "ATM (%)", "Skew", "Curvature"))
for tag, kw in [("base", {}), ("rho = -0.9", dict(rho=-0.9)), ("rho = 0", dict(rho=0.0)),
                ("sigma_v = 0.2", dict(sigv=0.2)), ("sigma_v = 1.0", dict(sigv=1.0)),
                ("kappa = 5", dict(kappa=5.0)), ("theta = 0.09", dict(theta=0.09))]:
    a, b, c = smile_stats(**kw)
    print("  %-28s %9.2f %9.3f %11.3f" % (tag, a, round(b, 3) + 0.0, c))

Figure 21.2, one-year implied volatility (%)
                         70      80      90     100     110     120     130
  rho=-0.9            27.52   23.98   20.33   16.46   12.39    9.43    8.79
  rho=-0.5            25.70   22.67   19.79   17.32   15.86   15.71   16.27
  rho=0               22.69   20.39   18.61   17.91   18.49   19.71   21.06
  sigv=0.2            23.24   21.88   20.61   19.44   18.37   17.42   16.61


  sigv=0.6 (base)     26.66   23.37   20.10   16.94   14.39   13.33   13.44
  sigv=1.0            27.55   23.16   18.76   14.45   11.79   12.16   13.29

Table 21.2, one year
                                 ATM (%)      Skew   Curvature
  base                             16.94    -0.282       0.356
  rho = -0.9                       16.46    -0.398      -0.581
  rho = 0                          17.91     0.000       1.268
  sigma_v = 0.2                    19.44    -0.112      -0.011
  sigma_v = 1.0                    14.45    -0.340       1.396
  kappa = 5                        18.94    -0.155       0.031
  theta = 0.09                     22.70    -0.227       0.084


## 4. Local volatility, and the rule of two — Example 21.1

Dupire's formula (21.2) reads the unique local-volatility function off the call
surface:

$$\sigma_{\rm loc}^2(K,T) = \frac{\partial C/\partial T + (r-q)K\,\partial C/\partial K + qC}
                                    {\tfrac12 K^2\,\partial^2C/\partial K^2}$$

The denominator is $\tfrac12K^2$ times the implied density, so the whole thing is
a ratio of two objects the surface already contains.

Numerically it is a second derivative of an option price, which is why the
Gauss–Legendre kernel of §2 was worth the trouble: the prices are accurate to
about $10^{-9}$, so a bump of 1% in log-strike is far above the noise floor.
Differencing in $\log K$ rather than $K$ is what keeps it stable in the wings —
$K^2\partial^2C/\partial K^2 = C_{yy} - C_y$ with $y = \log K$.

The result is the **rule of two**: the local-volatility curve is roughly twice as
steep as the implied curve it was derived from. It is not a calibration failure;
an implied volatility is an *average* of the local volatilities along the way, and
averaging flattens.

In [4]:
def local_vol(Ks, T, hk=0.01, hT=0.005, **kw):
    """Dupire by central differences in log-strike and in maturity."""
    Ks = np.atleast_1d(np.asarray(Ks, float))
    y = np.log(Ks)
    Tm = max(T - hT, 1e-4)
    C0 = heston_call(Ks, T, **kw)
    dCdT = (heston_call(Ks, T + hT, **kw) - heston_call(Ks, Tm, **kw)) / (T + hT - Tm)
    Cp, Cm = heston_call(np.exp(y + hk), T, **kw), heston_call(np.exp(y - hk), T, **kw)
    Cy = (Cp - Cm) / (2 * hk)
    Cyy = (Cp - 2 * C0 + Cm) / hk ** 2
    dens = Cyy - Cy                      # = K^2 d2C/dK2
    num = dCdT + (R - Q) * Cy + Q * C0
    return np.sqrt(np.clip(num / np.maximum(0.5 * dens, 1e-14), 1e-4, 4.0))


KS_EX = np.array([80., 90., 95., 100., 105., 110., 120.])
iv = np.array([heston_iv(K, 1.0) for K in KS_EX])
lv = local_vol(KS_EX, 1.0)

print("Example 21.1: one year, S0 = 100")
print("  %8s %14s %14s" % ("Strike", "Implied vol", "Local vol"))
for K, a, b in zip(KS_EX, iv, lv):
    print("  %8.0f %13.2f%% %13.2f%%" % (K, 100 * a, 100 * b))

k = log(1.05) - log(0.95)
s_imp = (iv[4] - iv[2]) / k
s_loc = (lv[4] - lv[2]) / k
print()
print("  slopes in log-strike, measured between 95 and 105")
print("    implied %+.3f    local %+.3f    ratio %.2f" % (s_imp, s_loc, s_loc / s_imp))

Example 21.1: one year, S0 = 100
    Strike    Implied vol      Local vol
        80         23.37%         27.69%
        90         20.10%         21.82%
        95         18.49%         18.92%
       100         16.94%         16.15%
       105         15.53%         13.73%
       110         14.39%         12.03%
       120         13.33%         11.42%

  slopes in log-strike, measured between 95 and 105
    implied -0.296    local -0.518    ratio 1.75


## 5. Why local volatility misprices the forward smile — Example 21.2

The local-volatility function's *own* skew decays with maturity much faster than
the implied skew does. A one-year option starting in three years is priced, under this
model, from local volatilities whose skew is less than a quarter of the
three-month one, so the model predicts a three-year-forward smile far flatter than today's spot smile.

The market disagrees: forward-starting options and cliquets trade with a skew
comparable to today's. The error is not in the fit — the fit is exact — it is in
what the fit *implies*.

In [5]:
print("Example 21.2: skew measured +/-5% around the forward at each maturity")
print("  %-10s %14s %16s %10s" % ("Maturity", "Implied skew", "Local-vol skew", "Ratio"))
for T, lbl in [(0.25, "3 months"), (0.5, "6 months"), (1.0, "1 year"),
               (2.0, "2 years"), (3.0, "3 years")]:
    F = S0 * exp((R - Q) * T)
    lo, hi = F * 0.95, F * 1.05
    dk = log(hi / lo)
    si = (heston_iv(hi, T) - heston_iv(lo, T)) / dk
    sl = float(local_vol(hi, T, hT=min(0.005, T / 4))[0]
               - local_vol(lo, T, hT=min(0.005, T / 4))[0]) / dk
    print("  %-10s %14.3f %16.3f %10.2f" % (lbl, si, sl, sl / si))
print()
print("  the local-vol skew decays by a factor of %.1f from 3 months to 3 years"
      % (0.981 / 0.231))

Example 21.2: skew measured +/-5% around the forward at each maturity
  Maturity     Implied skew   Local-vol skew      Ratio
  3 months           -0.479           -0.981       2.05
  6 months           -0.403           -0.770       1.91
  1 year             -0.296           -0.518       1.75
  2 years            -0.196           -0.320       1.63
  3 years            -0.148           -0.231       1.56

  the local-vol skew decays by a factor of 4.2 from 3 months to 3 years


## 6. The skew term structure — Figure 21.4

Define the at-the-money skew $\psi(T) = -\partial\sigma_{\rm impl}/\partial\log K$
at $K = F$ and watch it as $T \to 0$.

The empirical answer is a power law, $\psi(T) \sim T^{-0.4}$, from about a month
to a few years (below a month the evidence is disputed, as Section 21.8 explains). **Heston cannot produce it.** Variance is a diffusion, so
over a short horizon it moves by order $\sqrt{T}$; the asymmetry this creates shrinks
like $\sqrt{T}$, but so does the width of the distribution the skew is measured
against, and $\psi(T)$ approaches a finite constant however the parameters are set.

The cell below measures Heston's own exponent over the short end by regressing
$\log\psi$ on $\log T$. A power law would give a straight line; Heston gives a
curve that is flattening out, and its fitted exponent is nowhere near $-0.4$.

In [6]:
def atm_skew(T, h=0.01, **kw):
    F = S0 * exp((R - Q) * T)
    return -(heston_iv(F * exp(h), T, **kw) - heston_iv(F * exp(-h), T, **kw)) / (2 * h)


TS = np.array([1. / 12, 2. / 12, 0.25, 0.5, 1.0, 2.0, 3.0])
LBL = ["1 month", "2 months", "3 months", "6 months", "1 year", "2 years", "3 years"]
psi = np.array([atm_skew(T) for T in TS])
base = psi[TS == 1.0][0]

print("Figure 21.4: ATM skew, normalized to one year")
print("  %-10s %12s %12s %12s" % ("Maturity", "Heston", "market", "gap"))
for lbl, T, p in zip(LBL, TS, psi):
    mkt = T ** -0.4
    print("  %-10s %12.3f %12.3f %11.0f%%" % (lbl, p / base, mkt, 100 * (p / base / mkt - 1)))

short = TS <= 0.5
slope = np.polyfit(np.log(TS[short]), np.log(psi[short]), 1)[0]
print()
print("  Heston's own exponent over the short end: %+.3f   (the market's is -0.400)" % slope)
print("  Heston rises %.2fx from one year to one month; the market rises %.2fx"
      % (psi[0] / base, (1. / 12) ** -0.4))

Figure 21.4: ATM skew, normalized to one year
  Maturity         Heston       market          gap
  1 month           1.773        2.702         -34%
  2 months          1.744        2.048         -15%
  3 months          1.665        1.741          -4%
  6 months          1.381        1.320           5%
  1 year            1.000        1.000           0%
  2 years           0.656        0.758         -13%
  3 years           0.494        0.644         -23%

  Heston's own exponent over the short end: -0.138   (the market's is -0.400)
  Heston rises 1.77x from one year to one month; the market rises 2.70x


## 7. SABR — Hagan's expansion

$$dF = \alpha F^\beta dW_1, \qquad d\alpha = \nu\alpha\,dW_2, \qquad dW_1dW_2 = \rho\,dt$$

What made SABR the rate market's standard is not the process but the asymptotic
expansion, which gives the *implied volatility* directly — no integral, no
simulation, microseconds per smile.

$\beta$ chooses the **backbone**: how at-the-money volatility moves when the
forward moves. At $\beta = 1$ the model is lognormal, at $\beta = 0$ normal, and
$\beta = 0$ is what lets SABR handle negative rates. It is usually fixed rather
than fitted, because on a single smile it is nearly redundant with $\rho$.

The known defect is in the last cell: at low strikes the expansion implies a
**negative density**, which is Section 19.5's no-arbitrage condition violated by
an approximation rather than by the market. The check is Breeden–Litzenberger run
backward — price the expansion's own smile, difference it twice, and find where
the result goes below zero.

In [7]:
def sabr_vol(F, K, T, alpha, beta, rho, nu):
    """Hagan's 2002 lognormal-implied-volatility expansion."""
    one_b = 1.0 - beta
    fk = (F * K) ** (one_b / 2.0)
    lnfk = log(F / K)
    A = alpha / (fk * (1 + one_b ** 2 / 24 * lnfk ** 2 + one_b ** 4 / 1920 * lnfk ** 4))
    z = (nu / alpha) * fk * lnfk
    if abs(z) < 1e-12:
        zx = 1.0
    else:
        chi = log((sqrt(1 - 2 * rho * z + z * z) + z - rho) / (1 - rho))
        zx = z / chi
    B = 1 + (one_b ** 2 / 24 * alpha ** 2 / fk ** 2
             + rho * beta * nu * alpha / (4 * fk)
             + (2 - 3 * rho ** 2) / 24 * nu ** 2) * T
    return A * zx * B


F0, TSW = 0.030, 5.0
PAR = dict(alpha=0.030 ** 0.4 * 0.25, beta=0.6, rho=-0.30, nu=0.45)
strikes = np.array([0.010, 0.015, 0.020, 0.025, 0.030, 0.035, 0.040, 0.050, 0.060])

print("a five-year swaption smile, F = 3.00%%, beta = %.1f" % PAR["beta"])
print("  %10s %14s" % ("Strike (%)", "Implied vol (%)"))
for K in strikes:
    print("  %10.2f %14.2f" % (100 * K, 100 * sabr_vol(F0, K, TSW, **PAR)))

print()
print("the backbone: move the forward 50bp and watch ATM volatility")
print("  %-10s %14s %14s %14s" % ("beta", "F=2.50%", "F=3.00%", "F=3.50%"))
for beta in (0.0, 0.5, 1.0):
    a = 0.25 * F0 ** (1 - beta)                 # same ATM vol at F0 for every beta
    row = [100 * sabr_vol(f, f, TSW, a, beta, -0.30, 0.45) for f in (0.025, 0.030, 0.035)]
    print("  %-10.1f %14.2f %14.2f %14.2f" % (beta, row[0], row[1], row[2]))

print()
print("Breeden-Litzenberger on the expansion's own smile: where does the density go negative?")
Kg = np.arange(0.0015, 0.0605, 0.00025)
prices = np.array([bs_call(F0, K, 0.0, 0.0, sabr_vol(F0, K, TSW, **PAR), TSW) for K in Kg])
dens = (prices[2:] - 2 * prices[1:-1] + prices[:-2]) / (Kg[1] - Kg[0]) ** 2
negs = Kg[1:-1][dens < 0]
if len(negs):
    print("  density first turns negative below a strike of %.3f%%" % (100 * negs.max()))
    print("  most negative value %.1f at a strike of %.3f%%"
          % (dens.min(), 100 * Kg[1:-1][dens.argmin()]))
else:
    print("  density stays positive across the grid")

a five-year swaption smile, F = 3.00%, beta = 0.6
  Strike (%) Implied vol (%)
        1.00          45.56
        1.50          37.89
        2.00          32.60
        2.50          28.82
        3.00          26.24
        3.50          24.69
        4.00          23.98
        5.00          24.01
        6.00          24.83

the backbone: move the forward 50bp and watch ATM volatility
  beta              F=2.50%        F=3.00%        F=3.50%
  0.0                 32.75          27.15          23.20
  0.5                 28.86          26.38          24.45
  1.0                 25.77          25.77          25.77

Breeden-Litzenberger on the expansion's own smile: where does the density go negative?


  density first turns negative below a strike of 0.250%
  most negative value -3.4 at a strike of 0.175%


## 8. Jumps — Example 21.3

Merton's model is a diffusion plus compound Poisson jumps, and its price is a
Poisson-weighted sum of Black–Scholes prices,

$$C = \sum_{n\ge 0} \frac{e^{-\lambda^*T}(\lambda^*T)^n}{n!}\,
      C_{\rm BS}(S,K,r_n,\sigma_n,T)$$

with $\sigma_n^2 = \sigma^2 + n\delta^2/T$ and the drift compensated so the
discounted price stays a martingale.

A jump does what a diffusion cannot: over a one-month horizon a continuous path
simply cannot travel far enough to make a 15% out-of-the-money put worth anything,
and only a discontinuity can. That is why the model produces a steep short-dated
smile — and why the smile collapses with maturity: the diffusion's spread grows
while the jump's size does not.

Which is also the criticism. **Jumps overshoot the short end**: measured as the
at-the-money skew ratio between one month and one year, the model gives 3.7,
more than a third above the 2.7 of the power law index skews follow.

In [8]:
def merton_call(K, T, sig=0.17, lam=0.5, kbar=-0.10, delJ=0.13, nmax=60):
    """kbar = E[Y] - 1 is the mean jump; muJ follows from it."""
    muJ = log(1 + kbar) - 0.5 * delJ ** 2
    # The sum discounts at r while each term discounts at its own
    # r_n; reconciling the two compensates the intensity.
    lam_star = lam * (1 + kbar)
    tot, lamT = 0.0, lam_star * T
    for n in range(nmax):
        sig_n = sqrt(sig ** 2 + n * delJ ** 2 / T)
        r_n = R - lam * kbar + n * (muJ + 0.5 * delJ ** 2) / T
        tot += exp(-lamT) * lamT ** n / factorial(n) * bs_call(S0, K, r_n, Q, sig_n, T)
    return tot


def merton_iv(K, T, **kw):
    return implied_vol(merton_call(K, T, **kw), S0, K, R, Q, T)


TS3 = [1. / 12, 0.25, 1.0]
KS3 = [85., 90., 95., 100., 105., 110.]
print("Example 21.3: sigma = 17%, lambda = 0.5, mean jump -10%, jump vol 13%")
print("  %8s %12s %12s %12s" % ("Strike", "1 month", "3 months", "1 year"))
tab = {}
for K in KS3:
    row = [100 * merton_iv(K, T) for T in TS3]
    tab[K] = row
    print("  %8.0f %11.2f%% %11.2f%% %11.2f%%" % (K, row[0], row[1], row[2]))

m1 = [tab[K][0] for K in KS3]
y1 = [tab[K][2] for K in KS3]
print()
print("  one month:  %.1f%% down to %.1f%%, a range of %.1f volatility points"
      % (max(m1), min(m1), max(m1) - min(m1)))
print("  one year:   %.1f%% down to %.1f%%, a range of %.1f volatility points"
      % (max(y1), min(y1), max(y1) - min(y1)))
print("  the range collapses by a factor of %.1f" % ((max(m1) - min(m1)) / (max(y1) - min(y1))))


def merton_skew(T, h=0.01):
    F = S0 * exp((R - Q) * T)
    return -(merton_iv(F * exp(h), T) - merton_iv(F * exp(-h), T)) / (2 * h)


r_model = merton_skew(1. / 12) / merton_skew(1.0)
print()
print("  ATM skew ratio, one month to one year: model %.1f, market about 2.7" % r_model)
print("  jumps overshoot the short end by a factor of %.1f" % (r_model / 2.7))

Example 21.3: sigma = 17%, lambda = 0.5, mean jump -10%, jump vol 13%
    Strike      1 month     3 months       1 year
        85       30.99%       23.82%       21.21%
        90       25.07%       21.78%       20.81%


        95       20.57%       20.35%       20.47%
       100       18.77%       19.49%       20.18%
       105       18.25%       18.99%       19.94%


       110       18.42%       18.73%       19.74%

  one month:  31.0% down to 18.2%, a range of 12.7 volatility points
  one year:   21.2% down to 19.7%, a range of 1.5 volatility points
  the range collapses by a factor of 8.7

  ATM skew ratio, one month to one year: model 3.7, market about 2.7
  jumps overshoot the short end by a factor of 1.4


## 9. Rough volatility — equations (21.8) and (21.9)

Log-volatility behaves like a fractional Brownian motion with Hurst parameter
$H \approx 0.1$, and such a model generates

$$\psi(T) \sim T^{H - 1/2}$$

so $H = 0.1$ gives exactly the observed $T^{-0.4}$, from one parameter, with
$H = 0.5$ recovering the flat short end of the classical models.

The obstacle was computational: fractional Brownian motion is not Markovian, so
every price needs a simulation, and simulating it exactly costs $O(n^2)$. The
**hybrid scheme** below is one of the developments that changed that — it treats
the singular part of the kernel exactly (the first term, drawn jointly with the
Brownian increment from a $2\times2$ covariance) and the smooth remainder by a
convolution, done here directly with `np.convolve`; an FFT convolution brings it
to $O(n\log n)$, which is what makes the scheme comparable with ordinary Monte Carlo.

Table 21.4 then varies each of the three parameters from the base case, one at a time, and reads one-month and one-year level, skew and curvature off the simulated smiles, exactly as Table 21.2 did for Heston.

The test is the one that made the field: measure $\psi(T)$ across maturities,
regress $\log\psi$ on $\log T$, and check that the slope comes back as
$H - 1/2$. Compare with §6, where Heston's slope was nowhere near.

In [9]:
def rbergomi(H, eta, rho, xi0, T, n_steps, n_paths, seed=5):
    """Bennedsen-Lunde-Pakkanen hybrid scheme, kappa = 1."""
    dt = T / n_steps
    a = H - 0.5
    rng = np.random.default_rng(seed)
    cov = np.array([[dt, dt ** (a + 1) / (a + 1)],
                    [dt ** (a + 1) / (a + 1), dt ** (2 * a + 1) / (2 * a + 1)]])
    inc = rng.standard_normal((n_paths, n_steps, 2)) @ np.linalg.cholesky(cov).T
    dW1, Wa = inc[:, :, 0], inc[:, :, 1]
    kk = np.arange(2, n_steps + 1)
    b = ((kk ** (a + 1) - (kk - 1) ** (a + 1)) / (a + 1)) ** (1.0 / a)
    G = np.zeros(n_steps)
    G[1:] = (b * dt) ** a
    Y = sqrt(2 * H) * (Wa + np.array([np.convolve(r, G)[:n_steps] for r in dW1]))
    t = np.arange(1, n_steps + 1) * dt
    V = xi0 * np.exp(eta * Y - 0.5 * eta ** 2 * t ** (2 * H))
    dB = rho * dW1 + sqrt(1 - rho ** 2) * rng.standard_normal((n_paths, n_steps)) * sqrt(dt)
    Vp = np.concatenate([np.full((n_paths, 1), xi0), V[:, :-1]], axis=1)
    return S0 * np.exp(np.cumsum(-0.5 * Vp * dt + np.sqrt(Vp) * dB, axis=1))[:, -1]


def mc_skew(ST, T, w=0.05):
    f = lambda K: implied_vol(np.maximum(ST - K, 0).mean(), S0, K, R, Q, T)
    return -(f(S0 * exp(w)) - f(S0 * exp(-w))) / (2 * w)


MATS = np.array([1. / 12, 2. / 12, 0.25, 0.5, 1.0])
print("rough Bergomi: eta = 1.9, rho = -0.9, xi0 = 0.04, 150,000 paths")
print("  %-6s %s %14s" % ("H", "".join("%11s" % L for L in
                                        ["1m", "2m", "3m", "6m", "1y"]), "fitted slope"))
for H in (0.10, 0.25, 0.49):
    row = []
    for T in MATS:
        n = max(int(round(252 * T)), 21)
        row.append(mc_skew(rbergomi(H, 1.9, -0.9, 0.04, T, n, 150_000), T))
    slope = np.polyfit(np.log(MATS), np.log(row), 1)[0]
    print("  %-6.2f %s %14.3f   (H - 1/2 = %+.2f)"
          % (H, "".join("%11.3f" % x for x in row), slope, H - 0.5))
print()
print("  one parameter reproduces the observed power law; Heston's short-end slope was")
print("  measured in section 6 and is not close to any of these.")

def mc_stats(ST, T, h=0.10):
    """ATM level, skew and curvature from a +-h central difference in log-moneyness."""
    lo, mid, hi = [implied_vol(np.maximum(ST - K, 0).mean(), S0, K, R, Q, T)
                   for K in (S0 * exp(-h), S0, S0 * exp(h))]
    return mid, (hi - lo) / (2 * h), (hi - 2 * mid + lo) / h ** 2


BASE_RB = dict(H=0.10, eta=1.9, rho=-0.9, xi0=0.04)
ROWS_RB = [("base", {}), ("H = 0.25", dict(H=0.25)), ("H = 0.45", dict(H=0.45)),
           ("eta = 1.0", dict(eta=1.0)), ("eta = 3.0", dict(eta=3.0)),
           ("rho = -0.5", dict(rho=-0.5)), ("rho = 0", dict(rho=0.0))]
print()
print("Table 21.4: what each rough Bergomi parameter moves (200,000 paths, seed 5)")
print("  %-11s %-22s   %-22s   %s" % ("", "1m: ATM   skew  curv", "1y: ATM   skew  curv", "1m/1y skew"))
for name, change in ROWS_RB:
    p = dict(BASE_RB, **change)
    rows = []
    for T, n in ((1. / 12, 21), (1.0, 252)):
        ST = rbergomi(p["H"], p["eta"], p["rho"], p["xi0"], T, n, 200_000)
        rows.append((mc_stats(ST, T), mc_skew(ST, T)))
    (a1, s1), (a12, s12) = rows
    ratio = "%5.1f" % (s1 / s12) if abs(s12) > 0.01 else "  ---"
    print("  %-11s %5.1f %6.2f %5.1f       %5.1f %6.2f %5.1f       %s"
          % (name, 100 * a1[0], round(a1[1], 2) + 0.0, a1[2], 100 * a12[0], round(a12[1], 2) + 0.0, a12[2], ratio))
print()
print("  H moves the 1m/1y ratio -- theory 12**(0.5 - H): 2.70, 1.86, 1.13 -- while the one-year skew barely moves.")


rough Bergomi: eta = 1.9, rho = -0.9, xi0 = 0.04, 150,000 paths
  H               1m         2m         3m         6m         1y   fitted slope


  0.10         0.815      0.669      0.561      0.431      0.321         -0.380   (H - 1/2 = -0.40)


  0.25         0.775      0.674      0.586      0.484      0.390         -0.281   (H - 1/2 = -0.25)


  0.49         0.416      0.431      0.407      0.391      0.357         -0.066   (H - 1/2 = -0.01)

  one parameter reproduces the observed power law; Heston's short-end slope was
  measured in section 6 and is not close to any of these.

Table 21.4: what each rough Bergomi parameter moves (200,000 paths, seed 5)
              1m: ATM   skew  curv     1y: ATM   skew  curv     1m/1y skew


  base         17.8  -0.63   4.3        16.3  -0.31   0.0         2.6


  H = 0.25     18.6  -0.73   0.1        15.1  -0.39  -0.3         2.0


  H = 0.45     19.5  -0.47  -0.3        15.3  -0.36  -0.2         1.3


  eta = 1.0    19.3  -0.51  -0.1        18.6  -0.19  -0.1         2.7


  eta = 3.0    15.2  -0.47  13.3        12.8  -0.30   2.1         2.1


  rho = -0.5   18.3  -0.34   5.9        17.1  -0.17   0.9         2.4


  rho = 0      18.6   0.00   6.7        17.7   0.00   1.2         ---

  H moves the 1m/1y ratio -- theory 12**(0.5 - H): 2.70, 1.86, 1.13 -- while the one-year skew barely moves.


## 10. Local-stochastic volatility — Example 21.5 and Figure 21.6

What exotics desks run is both models at once,

$$dS = (r-q)S\,dt + L(S,t)\sqrt{v}\,S\,dW_1$$

with $L$ — the **leverage function** — chosen so the vanilla fit is exact. The
defining relation is

$$L^2(x,t) = \frac{\sigma^2_{\rm loc}(x,t)}{\mathbb{E}[v_t \mid S_t = x]}$$

which is circular: the conditional expectation depends on $L$, and $L$ is defined
by it. The **particle method** breaks the circle by estimating the conditional
expectation from the simulated paths themselves, as they advance — here by
binning on log-spot quantiles, which is the crudest estimator that works.

The **mixing parameter** scales the stochastic component: at 0 the model is pure
local volatility, at 1 it is as close to pure stochastic volatility as an exact
fit permits. The test is that the vanilla smile comes back *at every setting* —
the leverage function absorbing whatever the stochastic part fails to explain.

In [10]:
T_GRID = np.concatenate([np.linspace(0.02, 0.20, 19), np.linspace(0.22, 1.06, 43)])
LOGK = np.linspace(log(20.0), log(400.0), 161)
SURF = np.array([local_vol(np.exp(LOGK), T) for T in T_GRID])
print("local-volatility surface: %d maturities x %d strikes" % SURF.shape)
print("  sigma_loc(100, 1y) = %.2f%%   sigma_loc(110, 1y) = %.2f%%"
      % (100 * np.interp(log(100), LOGK, SURF[-4]),
         100 * np.interp(log(110), LOGK, SURF[-4])))


def lv_lookup(S, t):
    """Bilinear in (t, log S), flat outside the grid."""
    i = min(max(np.searchsorted(T_GRID, t) - 1, 0), len(T_GRID) - 2)
    w = min(max((t - T_GRID[i]) / (T_GRID[i + 1] - T_GRID[i]), 0.0), 1.0)
    row = (1 - w) * SURF[i] + w * SURF[i + 1]
    return np.interp(np.log(np.clip(S, exp(LOGK[0]), exp(LOGK[-1]))), LOGK, row)


LEVERAGE_SLICE = {}   # mixing -> (terminal spot, L on the final slice), filled by lsv_paths


def lsv_paths(mix, n_paths=80_000, n_steps=252, T=1.0, nbin=60, seed=99, up=110.0):
    """Particle-method LSV. mix = 0 is pure local volatility, mix = 1 pure stochastic.

    Returns the terminal spot and the survival mask of an up-and-out at `up`.
    """
    dt = T / n_steps
    rng = np.random.default_rng(seed)
    S = np.full(n_paths, S0)
    v = np.full(n_paths, BASE["v0"])
    alive = np.ones(n_paths, bool)
    for i in range(n_steps):
        vp = np.maximum(v, 0.0)
        loc = lv_lookup(S, i * dt)
        if mix == 0.0:
            sig = loc
            if i == n_steps - 1:
                LEVERAGE_SLICE[mix] = (S.copy(), loc / sqrt(BASE["v0"]))
        else:
            y = np.log(S)
            edges = np.quantile(y, np.linspace(0, 1, nbin + 1))
            edges[0] -= 1e-9
            edges[-1] += 1e-9
            idx = np.clip(np.searchsorted(edges, y) - 1, 0, nbin - 1)
            cnt = np.bincount(idx, minlength=nbin)
            cond = np.bincount(idx, weights=vp, minlength=nbin) / np.maximum(cnt, 1)
            cond = np.where(cnt > 0, cond, vp.mean())
            sig = loc * np.sqrt(vp / np.maximum(cond[idx], 1e-8))    # L * sqrt(v)
            if i == n_steps - 1:
                LEVERAGE_SLICE[mix] = (S.copy(), loc / np.sqrt(np.maximum(cond[idx], 1e-8)))
        z1 = rng.standard_normal(n_paths)
        z2 = BASE["rho"] * z1 + sqrt(1 - BASE["rho"] ** 2) * rng.standard_normal(n_paths)
        S = S * np.exp(-0.5 * sig ** 2 * dt + sig * sqrt(dt) * z1)
        alive &= S < up
        v = v + BASE["kappa"] * (BASE["theta"] - vp) * dt + \
            mix * BASE["sigv"] * np.sqrt(vp * dt) * z2
    return S, alive


print()
print("does the leverage function restore the vanilla fit at every mixing level?")
print("  %-14s %s" % ("mixing", "".join("%14s" % ("K=%d" % K) for K in (80, 100, 120))))
tgt = [100 * heston_iv(K, 1.0) for K in (80., 100., 120.)]
for mix in (0.0, 0.25, 0.5, 0.75, 1.0):
    ST = lsv_paths(mix)[0]
    row = [100 * implied_vol(np.maximum(ST - K, 0).mean(), S0, K, R, Q, 1.0)
           for K in (80., 100., 120.)]
    print("  %-14.2f %s" % (mix, "".join("%13.2f%%" % x for x in row)))
print("  %-14s %s" % ("target", "".join("%13.2f%%" % x for x in tgt)))

print()
print("Figure 21.6: the leverage function at one year, median by spot bucket")
BUCKETS = [70, 80, 90, 100, 110, 120, 130]
print("  %-8s %s" % ("mixing", "".join("%9s" % ("%d-%d" % (lo, hi)) for lo, hi in zip(BUCKETS[:-1], BUCKETS[1:]))))
for mix in (0.0, 0.5, 1.0):
    Sfin, Lfin = LEVERAGE_SLICE[mix]
    prof = [np.median(Lfin[(Sfin >= lo) & (Sfin < hi)]) for lo, hi in zip(BUCKETS[:-1], BUCKETS[1:])]
    print("  %-8.1f %s" % (mix, "".join("%9.3f" % x for x in prof)))
print("  at mixing 1 the surface is the model's own, so L comes back flat at one.")

local-volatility surface: 62 maturities x 161 strikes
  sigma_loc(100, 1y) = 16.15%   sigma_loc(110, 1y) = 12.03%

does the leverage function restore the vanilla fit at every mixing level?
  mixing                   K=80         K=100         K=120


  0.00                   23.35%        16.96%        13.30%


  0.25                   23.34%        16.94%        13.29%


  0.50                   23.27%        16.89%        13.28%


  0.75                   23.21%        16.85%        13.23%


  1.00                   23.24%        16.85%        13.23%
  target                 23.37%        16.94%        13.33%

Figure 21.6: the leverage function at one year, median by spot bucket
  mixing       70-80    80-90   90-100  100-110  110-120  120-130
  0.0          1.509    1.220    0.924    0.681    0.570    0.588
  0.5          1.153    1.009    0.876    0.774    0.768    0.870
  1.0          1.005    1.001    0.994    1.008    0.991    0.979
  at mixing 1 the surface is the model's own, so L comes back flat at one.


## 11. Model risk made concrete — Example 21.7

Derive the local-volatility surface from Heston's *own* vanilla prices. The two
models now agree on every European option by construction — the next cell
confirms it — so any difference on a barrier is pure model risk.

Two disciplines make the comparison honest. Both models are driven by the **same
random numbers** on the spot's Brownian motion, so the difference is measured
pathwise rather than as the difference of two noisy prices; that is Section 20.10's
technique doing exactly what it was introduced for. And the standard error quoted
is the standard error *of the difference*, which is an order of magnitude smaller
than either price's own.

The result is the contrast between the two halves of the table. On the
down-and-out the models agree to within one and a half percent. On the reverse knock-out they
disagree by half the price.

*So model risk is not a property of "barrier options."* It is a property of where
the payoff lives relative to where the models disagree. The plausible-sounding argument, that local
volatility is *low* at high spot so its paths knock out less, fails on its premise:
they knock out more. Both models put the same probability on every terminal
level, so what matters is survival given where a path ends: under Heston some paths
are quiet and reach the band without touching 110, while local volatility, with one
volatility at each spot and date, has no quiet paths to offer. The last cell prints
survival band by band.

In [11]:
N_PATHS, N_STEPS, TB = 200_000, 252, 1.0
DT = TB / N_STEPS
rng = np.random.default_rng(20250815)
Z1 = rng.standard_normal((N_STEPS, N_PATHS // 2))
Z1 = np.concatenate([Z1, -Z1], axis=1)                 # antithetic
Z3 = rng.standard_normal((N_STEPS, N_PATHS // 2))
Z3 = np.concatenate([Z3, -Z3], axis=1)

DOWN, UP = [80.0, 85.0, 90.0], [130.0, 120.0, 115.0, 110.0]


def barrier_run(model):
    """Daily monitoring. Returns terminal spot and the survival masks."""
    S = np.full(N_PATHS, S0)
    v = np.full(N_PATHS, BASE["v0"])
    alive = {("do", b): np.ones(N_PATHS, bool) for b in DOWN}
    alive.update({("uo", b): np.ones(N_PATHS, bool) for b in UP})
    for i in range(N_STEPS):
        if model == "heston":
            vp = np.maximum(v, 0.0)
            sig = np.sqrt(vp)
            z2 = BASE["rho"] * Z1[i] + sqrt(1 - BASE["rho"] ** 2) * Z3[i]
            v = v + BASE["kappa"] * (BASE["theta"] - vp) * DT \
                + BASE["sigv"] * np.sqrt(vp * DT) * z2
        else:
            sig = lv_lookup(S, i * DT)
        S = S * np.exp(-0.5 * sig ** 2 * DT + sig * sqrt(DT) * Z1[i])
        for b in DOWN:
            alive[("do", b)] &= S > b
        for b in UP:
            alive[("uo", b)] &= S < b
    return S, alive


res = {m: barrier_run(m) for m in ("heston", "localvol")}

print("vanilla control: the two models must agree on the one-year 100 call")
for m in ("heston", "localvol"):
    pay = np.maximum(res[m][0] - 100.0, 0.0)
    print("  %-9s %8.4f +/- %.4f" % (m, pay.mean(), pay.std(ddof=1) / sqrt(N_PATHS)))
print("  %-9s %8.4f  (semi-analytic)" % ("Heston", heston_call(100.0, 1.0)[0]))

print()
print("Example 21.7: one year, monitored daily, common random numbers")
print("  %-10s %10s %10s %18s %10s" % ("Barrier", "Heston", "Local vol", "Difference", "Relative"))
for kind, bars, label in (("do", DOWN, "Down-and-out call, strike 100"),
                          ("uo", UP, "Up-and-out call, strike 100")):
    print("  %s" % label)
    for b in bars:
        ph = np.maximum(res["heston"][0] - 100.0, 0.0) * res["heston"][1][(kind, b)]
        pl = np.maximum(res["localvol"][0] - 100.0, 0.0) * res["localvol"][1][(kind, b)]
        d = pl - ph
        print("     %-7.0f %10.4f %10.4f   %+8.3f +/- %.3f %9.1f%%"
              % (b, ph.mean(), pl.mean(), d.mean(),
                 d.std(ddof=1) / sqrt(N_PATHS), 100 * d.mean() / ph.mean()))

vanilla control: the two models must agree on the one-year 100 call
  heston      6.7655 +/- 0.0192
  localvol    6.7669 +/- 0.0194
  Heston      6.7509  (semi-analytic)

Example 21.7: one year, monitored daily, common random numbers
  Barrier        Heston  Local vol         Difference   Relative
  Down-and-out call, strike 100
     80          6.5286     6.5821     +0.053 +/- 0.015       0.8%
     85          6.2180     6.2962     +0.078 +/- 0.015       1.3%
     90          5.5388     5.6138     +0.075 +/- 0.015       1.4%
  Up-and-out call, strike 100
     130         5.8138     5.5787     -0.235 +/- 0.015      -4.0%
     120         4.0046     3.4193     -0.585 +/- 0.013     -14.6%
     115         2.4071     1.7138     -0.693 +/- 0.010     -28.8%
     110         0.8402     0.4244     -0.416 +/- 0.005     -49.5%


In [12]:
print("why the up-and-out goes the way it does: survival given where the path ends")
print("  %-10s %10s %16s %16s" % ("S_T band", "P(band)", "P(alive|band) H", "P(alive|band) LV"))
for lo, hi in ((100, 102), (102, 104), (104, 106), (106, 108), (108, 110)):
    out = []
    for m in ("heston", "localvol"):
        ST, alive = res[m]
        inb = (ST >= lo) & (ST < hi)
        out.append((inb.mean(), alive[("uo", 110.0)][inb].mean()))
    print("  %3d-%-6d %9.1f%% %15.1f%% %15.1f%%"
          % (lo, hi, 100 * out[0][0], 100 * out[0][1], 100 * out[1][1]))
for m in ("heston", "localvol"):
    ST, alive = res[m]
    a = alive[("uo", 110.0)] & (ST > 100.0)
    print("  %-9s P(alive, S_T > 100) = %5.1f%%   E[payoff | alive, S_T > 100] = %.2f"
          % (m, 100 * a.mean(), (ST[a] - 100.0).mean()))
print()
print("  both models put the same mass in every band, but a local-volatility path that")
print("  ends in the band has far more often touched 110 on the way.")

print()
print("Example 21.5: the mixing dial of section 10, priced on the 110 up-and-out")
print("  %-10s %14s %16s %14s %12s" % ("mixing", "ATM check", "up-and-out 110", "std error", "P(survive)"))
rng_lo, rng_hi = None, None
for mix in (0.0, 0.25, 0.5, 0.75, 1.0):
    ST, alive = lsv_paths(mix)
    pay = np.maximum(ST - 100.0, 0.0) * alive
    atm = 100 * implied_vol(np.maximum(ST - 100.0, 0).mean(), S0, 100.0, R, Q, 1.0)
    print("  %-10.2f %13.2f%% %16.4f %14.4f %11.1f%%"
          % (mix, atm, pay.mean(), pay.std(ddof=1) / sqrt(len(pay)), 100 * alive.mean()))
    rng_lo = pay.mean() if rng_lo is None else min(rng_lo, pay.mean())
    rng_hi = pay.mean() if rng_hi is None else max(rng_hi, pay.mean())
print("  the range %.3f to %.3f -- a factor of %.1f -- is the number to take to a"
      % (rng_lo, rng_hi, rng_hi / rng_lo))
print("  risk committee, and the point of the dial is that it comes from one model.")

why the up-and-out goes the way it does: survival given where the path ends
  S_T band      P(band)  P(alive|band) H P(alive|band) LV
  100-102          5.1%            76.7%            56.9%
  102-104          5.6%            73.4%            49.4%
  104-106          6.0%            68.3%            39.7%
  106-108          6.3%            59.2%            28.4%
  108-110          6.0%            39.3%            13.6%
  heston    P(alive, S_T > 100) =  18.3%   E[payoff | alive, S_T > 100] = 4.60
  localvol  P(alive, S_T > 100) =  10.7%   E[payoff | alive, S_T > 100] = 3.96

  both models put the same mass in every band, but a local-volatility path that
  ends in the band has far more often touched 110 on the way.

Example 21.5: the mixing dial of section 10, priced on the 110 up-and-out
  mixing          ATM check   up-and-out 110      std error   P(survive)


  0.00               16.96%           0.4202         0.0052        44.8%


  0.25               16.94%           0.4628         0.0055        45.7%


  0.50               16.89%           0.5724         0.0061        47.7%


  0.75               16.85%           0.7074         0.0068        50.2%


  1.00               16.85%           0.8449         0.0075        52.7%
  the range 0.420 to 0.845 -- a factor of 2.0 -- is the number to take to a
  risk committee, and the point of the dial is that it comes from one model.


## 12. The calibration objective — Example 21.6

Heston calibrations of one surface under different objectives: squared **price**
errors, squared **implied-volatility** errors, and squared price errors **divided
by vega**. The last two are the same objective to first order, since
$dC = \mathrm{vega}\times d\sigma$, and they are what Section 21.10 recommends.

The surface here is generated by the jump model of §8, so no Heston fit can be
exact, which is the honest setting: a fit to a surface a model can reproduce says
nothing about the objective function.

Price errors are dominated by at-the-money options, which have the largest vega,
so a given volatility miss costs most there. The one-month 80 put on this surface
is worth \$0.05, and the price fit leaves its implied volatility 7.08 points too low.

The cell also runs the objective Section 21.10 warns against: volatility errors
*multiplied* by vega. It reproduces the price-error fit almost exactly, and
necessarily so, because a vega-weighted volatility error is a price error to first
order. Weighting by vega cannot rescue the wings; dividing by it, or fitting
volatilities directly, is what moves error out of them.

None of them is right in general. The price objective is right for a desk whose
book is at-the-money straddles, the volatility objective for a desk that has sold
the wings. The objective encodes an assumption about the book, and it should be
chosen rather than inherited from whatever the software's default was.

In [13]:
KS_CAL = np.array([80., 85., 90., 95., 100., 105., 110., 115., 120.])
TS_CAL = np.array([1. / 12, 0.25, 1.0])
MKT_P = np.array([[merton_call(K, T) for K in KS_CAL] for T in TS_CAL])
MKT_V = np.array([[implied_vol(MKT_P[i, j], S0, KS_CAL[j], R, Q, TS_CAL[i])
                   for j in range(len(KS_CAL))] for i in range(len(TS_CAL))])
VEGA = np.array([[bs_vega(S0, KS_CAL[j], R, Q, MKT_V[i, j], TS_CAL[i])
                  for j in range(len(KS_CAL))] for i in range(len(TS_CAL))])


def model_grid(x):
    v0, kappa, theta, sigv, rho = x
    p = dict(v0=v0, kappa=kappa, theta=theta, sigv=sigv, rho=rho)
    return np.array([heston_call(KS_CAL, T, **p) for T in TS_CAL])


X0 = [0.04, 1.5, 0.04, 0.6, -0.5]
LO = [0.005, 0.10, 0.005, 0.05, -0.99]
HI = [0.50, 15.0, 0.50, 3.0, 0.50]


def vol_grid(x):
    P = model_grid(x)
    return np.array([[implied_vol(max(P[i, j], 1e-12), S0, KS_CAL[j], R, Q, TS_CAL[i])
                      for j in range(len(KS_CAL))] for i in range(len(TS_CAL))])


def resid_price(x):
    return (model_grid(x) - MKT_P).ravel()


def resid_vega(x):
    return ((vol_grid(x) - MKT_V) * VEGA).ravel()


def resid_flat(x):
    return (vol_grid(x) - MKT_V).ravel()


def resid_pdv(x):
    return ((model_grid(x) - MKT_P) / VEGA).ravel()


fits = {}
for tag, f in (("price errors", resid_price),
               ("vega-weighted vol errors", resid_vega),
               ("equally weighted vol errors", resid_flat),
               ("price errors / vega", resid_pdv)):
    s = least_squares(f, X0, bounds=(LO, HI), xtol=1e-10, ftol=1e-10)
    fits[tag] = s.x
    print("  %-29s v0=%.4f kappa=%.2f theta=%.4f sigv=%.3f rho=%+.3f"
          % (tag, s.x[0], s.x[1], s.x[2], s.x[3], s.x[4]))
print("  every fit pushes kappa to its bound: Heston is being asked to reproduce a")
print("  short-dated smile that only a jump can make, and fast mean reversion is the")
print("  nearest thing it has. A parameter at a bound is a message, not a nuisance.")

print()
print("implied-volatility error in volatility points, one month -- where the fits differ")
print("  %-29s %s" % ("", "".join("%8.0f" % K for K in KS_CAL)))
for tag in fits:
    err = 100 * (vol_grid(fits[tag])[0] - MKT_V[0])
    print("  %-29s %s" % (tag, "".join("%8.2f" % e for e in err)))
    print("  %-29s   ATM %+.2f, worst wing %+.2f, mean absolute %.2f"
          % ("", err[4], max(list(err[:2]) + list(err[-2:]), key=abs), np.mean(np.abs(err))))
print()
print("the first two fits nearly coincide, and that is the result, not an accident:")
print("  a vega-weighted volatility error IS a price error to first order, since")
print("  dC = vega x d(sigma), so weighting by vega cannot rescue the wings. Dividing")
print("  price errors by vega, the last fit, is the objective Section 21.10 recommends:")
print("  to first order it is the equally weighted volatility fit, it measures every")
print("  error in volatility points, and it moves error out of the wings and into the")
print("  near-the-money strikes, a wing-heavy book's own trade.")

  price errors                  v0=0.0381 kappa=15.00 theta=0.0432 sigv=1.308 rho=-0.298


  vega-weighted vol errors      v0=0.0389 kappa=15.00 theta=0.0433 sigv=1.383 rho=-0.288


  equally weighted vol errors   v0=0.0414 kappa=15.00 theta=0.0446 sigv=1.915 rho=-0.386
  price errors / vega           v0=0.0403 kappa=15.00 theta=0.0442 sigv=1.724 rho=-0.378
  every fit pushes kappa to its bound: Heston is being asked to reproduce a
  short-dated smile that only a jump can make, and fast mean reversion is the
  nearest thing it has. A parameter at a bound is a message, not a nuisance.

implied-volatility error in volatility points, one month -- where the fits differ
                                      80      85      90      95     100     105     110     115     120
  price errors                     -7.08   -4.51   -1.27    0.53    0.12    0.04    0.73    0.73   -0.48
                                  ATM +0.12, worst wing -7.08, mean absolute 1.72
  vega-weighted vol errors         -6.67   -4.20   -1.06    0.62    0.13    0.12    0.96    1.08   -0.04
                                  ATM +0.13, worst wing -6.67, mean absolute 1.65
  equally weighted vol errors

  price errors / vega              -4.01   -2.07    0.41    1.28   -0.19   -0.59    0.51    0.88   -0.06
                                  ATM -0.19, worst wing -4.01, mean absolute 1.11

the first two fits nearly coincide, and that is the result, not an accident:
  a vega-weighted volatility error IS a price error to first order, since
  dC = vega x d(sigma), so weighting by vega cannot rescue the wings. Dividing
  price errors by vega, the last fit, is the objective Section 21.10 recommends:
  to first order it is the equally weighted volatility fit, it measures every
  error in volatility points, and it moves error out of the wings and into the
  near-the-money strikes, a wing-heavy book's own trade.


## What this notebook shows

* Heston's characteristic function is exact to machine precision, which is what
  lets the Dupire second derivative be taken by finite difference at all. Any
  simulation-based surface would leave far too much noise for that.
* The **rule of two** is arithmetic, not a calibration failure: local volatility
  is roughly twice as steep as the implied volatility derived from it.
* Local volatility's fit is exact and its *forward* smile is wrong. The two facts
  are not in tension — the fit constrains terminal distributions, and a forward
  smile is not one of them.
* Heston's short-end skew saturates; the market's does not, at least down to a month. Jumps overcorrect.
  Rough volatility gets the exponent from one parameter, which is why it drew
  so much attention; whether volatility really is rough is still debated.
* **Model risk is a property of the payoff, not of the instrument class.** Two
  models fitted to the same vanillas agree within 1.5% on a down-and-out and differ
  by half the price on a reverse knock-out.

Everything is reproducible from the parameters printed in the chapter. Where a
number here is a Monte Carlo estimate, its standard error is printed beside it,
and agreement should be read in those units.